# Generation: StyleGAN2D
- Uses the '30kds_real_face_crop_dlib' dataset. This dataset has a more robust crop method explained on the 'create_30k_real_face_crop_dlib.py' file
- Images are all with size 128x128px
- Runs on gpu/cpu

In [1]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.utils import save_image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import torchvision.utils as vutils
import csv

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Parameters
batch_size = 32
latent_dim = 512  # StyleGAN uses higher-dimensional latent space
style_dim = 512
num_epochs = 250
learning_rate = 0.0002
beta1 = 0.5
beta2 = 0.999
num_train_images = 0  # Number of training images to use (0 to use all)
dataset_path = "30kds_real_face_crop_dlib"
save_interval = 10
image_size = 128

# Ensure directories exist for saving outputs
base_dir = os.getcwd()
model_dir = gen_images_dir = os.path.join(base_dir, "generators/styleGAN")
os.makedirs(gen_images_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# Prepare CSV file to log losses
loss_log_path = os.path.join(model_dir, "training_losses.csv")
with open(loss_log_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['epoch', 'discriminator_Loss', 'generator_Loss', 'r1_penalty'])

Using device: cuda


In [2]:
# Data Loading
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),  # match Tanh range (-1 to 1)
])

# Load the dataset
full_dataset = ImageFolder(root=dataset_path, transform=transform)

# Randomly select a subset of images
total_images = len(full_dataset)
if num_train_images == 0:
    dataset = full_dataset
    print(f"Full dataset loaded.")
elif total_images > num_train_images:
    indices = torch.randperm(total_images)[:num_train_images].tolist()
    dataset = Subset(full_dataset, indices)
    print(f"Loaded {num_train_images} images.")
else:
    dataset = full_dataset
    print(f"Warning: Requested {num_train_images} images but only {total_images} are available.")
    print(f"Full dataset loaded.")

# Create data loader
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

Full dataset loaded.


In [3]:
# Weight initialization
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1 and hasattr(m, 'weight'):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1 and hasattr(m, 'weight'):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)
    elif classname.find('Linear') != -1 and hasattr(m, 'weight'):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Pixel-wise noise injection
class NoiseInjection(nn.Module):
    def __init__(self, channel):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(1, channel, 1, 1))
        
    def forward(self, image, noise=None):
        if noise is None:
            batch, _, height, width = image.shape
            noise = torch.randn(batch, 1, height, width, device=image.device)
            
        return image + self.weight * noise

# AdaIN (Adaptive Instance Normalization)
class AdaIN(nn.Module):
    def __init__(self, style_dim, channels):
        super().__init__()
        self.instance_norm = nn.InstanceNorm2d(channels)
        self.style_scale = nn.Linear(style_dim, channels)
        self.style_bias = nn.Linear(style_dim, channels)
        
    def forward(self, x, style):
        x = self.instance_norm(x)
        style = style.view(style.size(0), -1)
        scale = self.style_scale(style).unsqueeze(2).unsqueeze(3)
        bias = self.style_bias(style).unsqueeze(2).unsqueeze(3)
        return scale * x + bias

# StyleConv Block
class StyleConvBlock(nn.Module):
    def __init__(self, in_channel, out_channel, style_dim, upsample=False):
        super().__init__()
        self.upsample = upsample
        if upsample:
            self.conv1 = nn.ConvTranspose2d(in_channel, out_channel, 4, 2, 1, bias=False)
        else:
            self.conv1 = nn.Conv2d(in_channel, out_channel, 3, 1, 1, bias=False)
            
        self.noise1 = NoiseInjection(out_channel)
        self.adain1 = AdaIN(style_dim, out_channel)
        self.activation1 = nn.LeakyReLU(0.2, inplace=True)
        
        self.conv2 = nn.Conv2d(out_channel, out_channel, 3, 1, 1, bias=False)
        self.noise2 = NoiseInjection(out_channel)
        self.adain2 = AdaIN(style_dim, out_channel)
        self.activation2 = nn.LeakyReLU(0.2, inplace=True)
        
    def forward(self, x, style):
        if self.upsample:
            x = self.conv1(x)
        else:
            x = self.conv1(x)
            
        x = self.noise1(x)
        x = self.adain1(x, style)
        x = self.activation1(x)
        
        x = self.conv2(x)
        x = self.noise2(x)
        x = self.adain2(x, style)
        x = self.activation2(x)
        
        return x

# Mapping Network (z → w)
class MappingNetwork(nn.Module):
    def __init__(self, latent_dim, style_dim, n_mlp=8):
        super().__init__()
        layers = []
        layers.append(nn.Linear(latent_dim, style_dim))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        
        for _ in range(n_mlp - 1):
            layers.append(nn.Linear(style_dim, style_dim))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            
        self.mapping = nn.Sequential(*layers)
        
    def forward(self, z):
        w = self.mapping(z)
        return w

In [4]:
# Generator with Style
class StyleGenerator(nn.Module):
    def __init__(self, style_dim, latent_dim=512, n_mlp=8):
        super().__init__()
        
        # Mapping Network from z to w
        self.mapping = MappingNetwork(latent_dim, style_dim, n_mlp)
        
        # Initial constant input
        self.input = nn.Parameter(torch.randn(1, 512, 4, 4))
        
        # Style layers for different resolutions
        self.style_conv1 = StyleConvBlock(512, 512, style_dim, upsample=False)
        self.style_conv2 = StyleConvBlock(512, 512, style_dim, upsample=True)  # 8x8
        self.style_conv3 = StyleConvBlock(512, 256, style_dim, upsample=True)  # 16x16
        self.style_conv4 = StyleConvBlock(256, 128, style_dim, upsample=True)  # 32x32
        self.style_conv5 = StyleConvBlock(128, 64, style_dim, upsample=True)   # 64x64
        self.style_conv6 = StyleConvBlock(64, 32, style_dim, upsample=True)    # 128x128
        
        # To RGB layers
        self.to_rgb = nn.Sequential(
            nn.Conv2d(32, 3, 1, 1, 0, bias=True),
            nn.Tanh()
        )
        
    def forward(self, z, return_latents=False):
        # Map z to w space
        w = self.mapping(z)
        
        # Replicate the input for batch size
        batch_size = z.size(0)
        x = self.input.repeat(batch_size, 1, 1, 1)
        
        # Apply style modulation at each resolution
        x = self.style_conv1(x, w)
        x = self.style_conv2(x, w)
        x = self.style_conv3(x, w)
        x = self.style_conv4(x, w)
        x = self.style_conv5(x, w)
        x = self.style_conv6(x, w)
        
        # Convert to RGB
        images = self.to_rgb(x)
        
        if return_latents:
            return images, w
        else:
            return images

# Discriminator (Progressive structure similar to StyleGAN)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.progression = nn.Sequential(
            # 128x128 -> 64x64
            nn.Conv2d(3, 32, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 64x64 -> 32x32
            nn.Conv2d(32, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 32x32 -> 16x16
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 16x16 -> 8x8
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 8x8 -> 4x4
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 4x4 -> 1x1
            nn.Conv2d(512, 1, 4, 1, 0, bias=False)
        )
        
    def forward(self, x):
        out = self.progression(x)
        return out.view(out.size(0), -1)

# Initialize models
generator = StyleGenerator(style_dim, latent_dim).to(device)
discriminator = Discriminator().to(device)

generator.apply(weights_init)
discriminator.apply(weights_init)

Discriminator(
  (progression): Sequential(
    (0): Conv2d(3, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (6): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): LeakyReLU(negative_slope=0.2, inplace=True)
    (8): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (9): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.2, inplace=True)
    (11): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (12): BatchNorm2d(512, eps=1e-

In [5]:
# Loss and optimizers
criterion = nn.BCEWithLogitsLoss()
g_optimizer = optim.Adam(generator.parameters(), lr=learning_rate, betas=(beta1, beta2))
d_optimizer = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(beta1, beta2))

# Fixed noise for visualization
fixed_noise = torch.randn(16, latent_dim, device=device)

# R1 gradient penalty
def r1_penalty(real_pred, real_img):
    grad_real = torch.autograd.grad(
        outputs=real_pred.sum(), inputs=real_img, create_graph=True
    )[0]
    grad_penalty = grad_real.pow(2).view(grad_real.shape[0], -1).sum(1).mean()
    return grad_penalty

In [6]:
# Training function
def train_styleGAN(generator, discriminator, dataloader, criterion, g_optimizer, d_optimizer, 
                 num_epochs, device, latent_dim, save_interval, gen_images_dir, loss_log_path, r1_gamma=10.0):
    # Fixed noise for visualization
    fixed_noise = torch.randn(16, latent_dim, device=device)
    
    print("Starting StyleGAN training...")
    
    for epoch in range(num_epochs):
        # Initialize epoch stats
        epoch_d_loss = 0.0
        epoch_g_loss = 0.0
        epoch_r1_penalty = 0.0
        num_batches = 0
        
        # Create progress bar for better visibility
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        # Batch training loop
        for i, (images, _) in enumerate(pbar):
            batch_size = images.size(0)
            real_images = images.to(device)
            
            # Train Discriminator
            d_optimizer.zero_grad()
            
            # Compute R1 penalty (requires gradients for real images)
            real_images.requires_grad = True
            real_pred = discriminator(real_images)
            r1_loss = r1_penalty(real_pred, real_images)
            real_images.requires_grad = False
            
            # Real image loss
            real_labels = torch.ones(batch_size, 1, device=device)
            d_real_loss = criterion(real_pred, real_labels)
            
            # Fake image loss
            noise = torch.randn(batch_size, latent_dim, device=device)
            with torch.no_grad():
                fake_images = generator(noise)
            fake_pred = discriminator(fake_images)
            fake_labels = torch.zeros(batch_size, 1, device=device)
            d_fake_loss = criterion(fake_pred, fake_labels)
            
            # Total discriminator loss
            d_loss = d_real_loss + d_fake_loss + r1_gamma * r1_loss
            d_loss.backward()
            d_optimizer.step()
            
            # Train Generator
            g_optimizer.zero_grad()
            noise = torch.randn(batch_size, latent_dim, device=device)
            fake_images = generator(noise)
            fake_pred = discriminator(fake_images)
            g_loss = criterion(fake_pred, real_labels)
            g_loss.backward()
            g_optimizer.step()

            # Update statistics
            last_batch_d_loss = d_loss.item() - r1_gamma * r1_loss.item()  # Pure D loss without penalty
            last_batch_g_loss = g_loss.item()
            last_batch_r1 = r1_loss.item()
            
            epoch_d_loss += last_batch_d_loss
            epoch_g_loss += last_batch_g_loss
            epoch_r1_penalty += last_batch_r1
            num_batches += 1
            
            # Update progress bar with current batch loss
            pbar.set_postfix({
                'D Loss': f"{last_batch_d_loss:.4f}",
                'G Loss': f"{last_batch_g_loss:.4f}",
                'R1': f"{last_batch_r1:.4f}"
            })
        
        # Average losses
        avg_d_loss = epoch_d_loss / num_batches
        avg_g_loss = epoch_g_loss / num_batches
        avg_r1 = epoch_r1_penalty / num_batches
        
        # Log losses to CSV
        with open(loss_log_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([epoch + 1, avg_d_loss, avg_g_loss, avg_r1])
        
        # Print current epoch loss
        print(f"Epoch [{epoch+1}/{num_epochs}] - D Loss: {avg_d_loss:.4f}, G Loss: {avg_g_loss:.4f}, R1: {avg_r1:.4f}")
        
        # Generate and save images at checkpoints
        if (epoch + 1) % save_interval == 0 or epoch == 0:
            try:
                with torch.no_grad():
                    fake_samples = generator(fixed_noise).detach().cpu()
                    grid = vutils.make_grid(fake_samples, nrow=4, padding=2, normalize=True)
                    save_path = os.path.join(gen_images_dir, f"epoch_{epoch+1}.png")
                    save_image(grid, save_path)
                    print(f"Generated samples saved to {save_path}")
            except Exception as e:
                print(f"Error during saving: {str(e)}")
    
    # Save final models
    try:
        torch.save(generator.state_dict(), os.path.join(os.path.dirname(gen_images_dir), "stylegan_generator_final.pth"))
        torch.save(discriminator.state_dict(), os.path.join(os.path.dirname(gen_images_dir), "stylegan_discriminator_final.pth"))
        print("Final models saved successfully")
    except Exception as e:
        print(f"Error saving final models: {str(e)}")

train_styleGAN(
    generator=generator,
    discriminator=discriminator,
    dataloader=dataloader,
    criterion=criterion,
    g_optimizer=g_optimizer,
    d_optimizer=d_optimizer,
    num_epochs=num_epochs,
    device=device,
    latent_dim=latent_dim,
    save_interval=save_interval,
    gen_images_dir=gen_images_dir,
    loss_log_path=loss_log_path
)

Starting StyleGAN training...


Epoch 1/250: 100%|██████████| 809/809 [01:11<00:00, 11.29it/s, D Loss=1.2122, G Loss=1.3076, R1=0.0030] 


Epoch [1/250] - D Loss: 2.0189, G Loss: 1.0805, R1: 0.0757
Generated samples saved to c:\Users\danie\Desktop\UNI-4ano\DL\generators/styleGAN\epoch_1.png


Epoch 2/250: 100%|██████████| 809/809 [01:11<00:00, 11.35it/s, D Loss=1.1701, G Loss=0.9565, R1=0.0058]


Epoch [2/250] - D Loss: 1.4409, G Loss: 1.0194, R1: 0.0043


...

Epoch [248/250] - D Loss: 1.2897, G Loss: 0.7778, R1: 0.0045


Epoch 249/250: 100%|██████████| 809/809 [01:09<00:00, 11.66it/s, D Loss=1.6122, G Loss=0.6801, R1=0.0056]


Epoch [249/250] - D Loss: 1.2876, G Loss: 0.7802, R1: 0.0048


Epoch 250/250: 100%|██████████| 809/809 [01:10<00:00, 11.52it/s, D Loss=1.1498, G Loss=0.7111, R1=0.0042]

Epoch [250/250] - D Loss: 1.2861, G Loss: 0.7825, R1: 0.0047
Generated samples saved to c:\Users\danie\Desktop\UNI-4ano\DL\generators/styleGAN\epoch_250.png
Final models saved successfully


In [7]:
# Generate final images
def generate_samples(num_images=64, truncation=0.7):
    generator.eval()
    with torch.no_grad():
        # Sample from truncated latent space for better quality
        z = torch.randn(num_images, latent_dim, device=device)
        z = torch.clamp(z, -truncation, truncation)
        sample_images = generator(z).detach().cpu()

        grid = vutils.make_grid(sample_images, nrow=8, padding=2, normalize=True)
        final_sample_path = os.path.join(gen_images_dir, "final_samples.png")
        save_image(grid, final_sample_path)

    print(f"Final samples saved to {final_sample_path}")
    return sample_images

# Generate final images
generate_samples(64)

Final samples saved to c:\Users\danie\Desktop\UNI-4ano\DL\generators/styleGAN\final_samples.png


tensor([[[[ 0.2862,  0.3781,  0.1718,  ..., -0.2088, -0.1573, -0.0373],
          [ 0.6887,  0.6022,  0.4318,  ..., -0.1513, -0.1174, -0.1011],
          [ 0.7428,  0.6585,  0.5052,  ..., -0.1891, -0.1580, -0.1699],
          ...,
          [ 0.9639,  0.9870,  0.9884,  ...,  0.7119,  0.6015,  0.7633],
          [ 0.9766,  0.9918,  0.9897,  ...,  0.7469,  0.7045,  0.8891],
          [ 0.9493,  0.9852,  0.9820,  ...,  0.8683,  0.9078,  0.9362]],

         [[-0.1183, -0.0079, -0.0813,  ..., -0.4150, -0.3630, -0.3231],
          [ 0.5430,  0.3558,  0.2249,  ..., -0.3835, -0.3796, -0.3332],
          [ 0.5233,  0.2954,  0.3081,  ..., -0.4115, -0.3782, -0.3764],
          ...,
          [ 0.8013,  0.8200,  0.8037,  ...,  0.6766,  0.5489,  0.5512],
          [ 0.8610,  0.8712,  0.8318,  ...,  0.6328,  0.5414,  0.7677],
          [ 0.6980,  0.8119,  0.7690,  ...,  0.5751,  0.7371,  0.8005]],

         [[-0.5181, -0.2965, -0.2742,  ..., -0.6061, -0.5682, -0.5915],
          [ 0.4660,  0.1212,  

In [8]:
# Generate style mixing examples
def generate_style_mixing():
    generator.eval()
    with torch.no_grad():
        # Source and target latent codes
        src_z = torch.randn(4, latent_dim, device=device)
        tgt_z = torch.randn(4, latent_dim, device=device)
        
        # Generate source and target w vectors
        src_w = generator.mapping(src_z)
        tgt_w = generator.mapping(tgt_z)
        
        # Create a grid of mixed styles
        mixed_samples = []
        # Original source images
        src_images = generator(src_z)
        mixed_samples.append(src_images)
        
        # Original target images
        tgt_images = generator(tgt_z)
        mixed_samples.append(tgt_images)
        
        # Style mixing grid
        mixed_w = []
        for i in range(4):
            for j in range(4):
                # Copy source w but replace some style components with target
                w_mix = src_w.clone()
                w_mix[i] = tgt_w[j]
                mixed_w.append(w_mix)
        
        # Generate style mixed images by reconstructing from w
        # This is simplified - in a full implementation you would modify
        # the generator to accept w directly at different layers
        style_mixed_images = []
        for w in mixed_w:
            # Use w directly with the generator internals
            # For this simplified version, we convert back to z-space representation
            fake_img = generator(w, return_latents=False)
            style_mixed_images.append(fake_img)
        
        # Concatenate and create grid
        all_samples = torch.cat(mixed_samples + style_mixed_images, dim=0)
        grid = vutils.make_grid(all_samples, nrow=4, padding=2, normalize=True)
        mix_sample_path = os.path.join(gen_images_dir, "style_mixing.png")
        save_image(grid, mix_sample_path)
        
    print(f"Style mixing examples saved to {mix_sample_path}")

generate_style_mixing()

Style mixing examples saved to c:\Users\danie\Desktop\UNI-4ano\DL\generators/styleGAN\style_mixing.png
